# **06 - Huấn luyện mô hình phân loại bình luận tiêu cực tiếng Việt**

Notebook này thực hiện toàn bộ pipeline huấn luyện **6 mô hình** phân loại bình luận tiêu cực tiếng Việt (ViHSD) với 3 lớp:

| Label ID | Tên lớp | Ý nghĩa |
|----------|---------|----------|
| 0 | CLEAN | Không tiêu cực |
| 1 | OFFENSIVE | Tiêu cực / Xúc phạm |
| 2 | HATE | Thù ghét / Kích động |

**Các bước chính:**
1. Import thư viện & đọc dữ liệu
2. Feature Engineering (TF-IDF, BoW, SVD)
3. Huấn luyện 6 mô hình với GridSearchCV
4. Learning Curves & Epoch Curves
5. Error Analysis
6. Tổng hợp & lưu kết quả

In [1]:
!pip install imbalanced-learn


---
## **Phần 0: Import thư viện**

Import toàn bộ thư viện cần thiết cho pipeline:
- **pandas, numpy**: xử lý dữ liệu và tính toán số học
- **sklearn**: các mô hình ML, vectorizer, metrics, model selection
- **joblib**: lưu/load model dạng `.pkl`
- **json, pathlib**: lưu kết quả dạng JSON và quản lý đường dẫn file

In [2]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_recall_fscore_support,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, learning_curve
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_class_weight

from imblearn.over_sampling import SMOTE
warnings.filterwarnings("ignore")

---
## **Phần 0B: Cấu hình đường dẫn & hằng số**

Thiết lập các biến cấu hình chung:
- **ROOT_DIR**: thư mục gốc của dự án - tự động phát hiện bằng cách kiểm tra thư mục `data/` tồn tại ở CWD hay ở thư mục cha (vì VS Code đặt CWD là workspace root, còn Jupyter thuần đặt CWD là thư mục chứa notebook)
- **DATA_DIR**: thư mục chứa dữ liệu đã tiền xử lý (`data/`)
- **OUTPUT_DIR**: thư mục lưu model và kết quả (`output/model/`)
- **RANDOM_STATE**: seed cho reproducibility
- **LABELS / LABEL_NAMES**: mapping nhãn số → tên lớp

In [ ]:
_cwd = Path(".").resolve()
if (_cwd / "data").exists():
    ROOT_DIR = _cwd
elif (_cwd.parent / "data").exists():
    ROOT_DIR = _cwd.parent
else:
    raise FileNotFoundError(
        f"Không tìm thấy thư mục 'data/' từ CWD={_cwd}. "
        "Hãy đảm bảo mở notebook từ đúng thư mục dự án."
    )

DATA_DIR = ROOT_DIR / "data"
OUTPUT_DIR = ROOT_DIR / "output" / "model"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
LABELS = np.array([0, 1, 2])
LABEL_NAMES = {0: "CLEAN", 1: "OFFENSIVE", 2: "HATE"}

print(f"CWD       : {_cwd}")
print(f"ROOT_DIR  : {ROOT_DIR}")
print(f"DATA_DIR  : {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

---
## **Phần 0C: Định nghĩa các hàm tiện ích**

Tạo các hàm helper tái sử dụng trong suốt notebook:

- **`load_split(name)`**: Đọc file CSV dữ liệu theo tên split (`train`, `dev`, `test`), kiểm tra file tồn tại và cột cần thiết (`free_text_clean`, `label_id`)
- **`evaluate_predictions(y_true, y_pred)`**: Tính toàn bộ metrics đánh giá (accuracy, precision, recall, F1-weighted, F1-macro, per-class F1) từ nhãn thực và nhãn dự đoán
- **`run_grid_search(estimator, param_grid, X, y, cv)`**: Wrapper cho `GridSearchCV` với `scoring='f1_weighted'` và `n_jobs=-1` (tận dụng tất cả CPU cores)
- **`save_json(path, payload)`**: Lưu dictionary ra file JSON với encoding UTF-8
- **`soft_voting_predict_proba(models, X)`**: Tính trung bình xác suất từ nhiều mô hình để thực hiện soft voting (dùng cho Voting Ensemble)

In [4]:
def load_split(name: str) -> pd.DataFrame:
    """Đọc file CSV dữ liệu theo tên split."""
    path = DATA_DIR / f"{name}_clean.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing data file: {path}")
    df = pd.read_csv(path)
    if "free_text_clean" not in df.columns or "label_id" not in df.columns:
        raise ValueError("Expected columns: free_text_clean, label_id")
    return df


def evaluate_predictions(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Tính toàn bộ metrics đánh giá từ nhãn thực và nhãn dự đoán."""
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    per_class_f1 = {
        LABEL_NAMES[label]: report[str(label)]["f1-score"] for label in LABELS
    }
    return {
        "accuracy": acc,
        "precision_weighted": precision,
        "recall_weighted": recall,
        "f1_weighted": f1_weighted,
        "f1_macro": f1_macro,
        "per_class_f1": per_class_f1,
        "classification_report": report,
    }


def run_grid_search(estimator, param_grid, X, y, cv):
    """Wrapper cho GridSearchCV với scoring='f1_weighted'."""
    search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        cv=cv,
        scoring="f1_weighted",
        n_jobs=-1,
    )
    search.fit(X, y)
    return search


def save_json(path: Path, payload: dict) -> None:
    """Lưu dictionary ra file JSON."""
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


def soft_voting_predict_proba(models, X):
    """Tính trung bình xác suất từ nhiều mô hình (soft voting)."""
    probas = [model.predict_proba(X) for model in models]
    return np.mean(probas, axis=0)

---
## **Phần 1: Đọc dữ liệu**

Đọc 3 tập dữ liệu đã tiền xử lý:
- **train_clean.csv** (~22,510 mẫu): dùng để huấn luyện mô hình
- **dev_clean.csv** (~2,633 mẫu): dùng để điều chỉnh siêu tham số (validation)
- **test_clean.csv** (~6,527 mẫu): dùng để đánh giá cuối cùng

Cột dữ liệu:
- `free_text_clean`: văn bản đã qua tiền xử lý NLP
- `label_id`: nhãn phân loại (0 = CLEAN, 1 = OFFENSIVE, 2 = HATE)

Dùng `.astype(str)` để phòng trường hợp có giá trị NaN trong cột text.

In [5]:
TEENCODE_DICT = {
    "dell": "đéo", "del": "đéo", "đell": "đéo", "đel": "đéo",
    "loz": "lồn", "lon": "lồn", "lòn": "lồn", "l": "lồn",
    "coin card": "củ cặc", "cc": "củ cặc", "cức": "cứt",
    "ms": "mới", "bh": "bây giờ", "kb": "không biết",
    "kk": "cười", "haha": "cười", "đhs": "đéo hiểu sao",
    "dm": "địt mẹ", "đm": "địt mẹ", "dmm": "địt mẹ mày", "vcl": "vãi lồn",
    "vl": "vãi lồn", "vkl": "vãi lồn", "cl": "cái lồn", "clgt": "cái lồn gì thế",
    "đcm": "địt con mẹ", "dcm": "địt con mẹ"
}

def normalize_teencode(text: str) -> str:
    words = str(text).split()
    return " ".join([TEENCODE_DICT.get(w, w) for w in words])

train_df = load_split("train")
dev_df = load_split("dev")
test_df = load_split("test")

X_train = train_df["free_text_clean"].apply(normalize_teencode).astype(str).to_numpy()
X_dev = dev_df["free_text_clean"].apply(normalize_teencode).astype(str).to_numpy()
X_test = test_df["free_text_clean"].apply(normalize_teencode).astype(str).to_numpy()

y_train = train_df["label_id"].to_numpy()
y_dev = dev_df["label_id"].to_numpy()
y_test = test_df["label_id"].to_numpy()

print(f"Train: {len(train_df):,} mẫu")
print(f"Dev  : {len(dev_df):,} mẫu")
print(f"Test : {len(test_df):,} mẫu")
print(f"\nPhân phối nhãn (train):")
print(train_df["label_id"].value_counts().sort_index())

Train: 22,510 mẫu
Dev  : 2,633 mẫu
Test : 6,527 mẫu

Phân phối nhãn (train):
label_id
0    18473
1     1542
2     2495
Name: count, dtype: int64


---
## **Phần 2A: Feature Engineering - TF-IDF Vectorizer**

Chuyển đổi văn bản thành vector số bằng **TF-IDF** (Term Frequency – Inverse Document Frequency):

| Tham số | Giá trị | Giải thích |
|---------|---------|------------|
| `max_features` | 50,000 | Giữ lại 50K từ/cụm từ phổ biến nhất |
| `ngram_range` | (1, 2) | Sử dụng cả unigram và bigram |
| `sublinear_tf` | True | Áp dụng `1 + log(tf)` thay vì `tf` thuần, giúp giảm ảnh hưởng của từ lặp nhiều |
| `dtype` | float32 | Tiết kiệm RAM so với float64 mặc định |

TF-IDF được dùng làm input chính cho: **Logistic Regression, LinearSVC, SGDClassifier, VotingEnsemble**.

**Quan trọng**: Fit vectorizer trên tập train, sau đó chỉ `transform` cho dev/test để tránh data leakage.

In [6]:
print("Vectorizing text với TF-IDF (10K features)...")
tfidf = TfidfVectorizer(
    max_features=10000, ngram_range=(1, 2), sublinear_tf=True, dtype=np.float32
)
X_train_tfidf_raw = tfidf.fit_transform(X_train)
X_dev_tfidf = tfidf.transform(X_dev)
X_test_tfidf = tfidf.transform(X_test)

print("Áp dụng SMOTE cân bằng dữ liệu TF-IDF...")
smote = SMOTE(random_state=RANDOM_STATE)
X_train_tfidf, y_train_tfidf = smote.fit_resample(X_train_tfidf_raw, y_train)

joblib.dump(tfidf, OUTPUT_DIR / "tfidf_vectorizer.pkl")

print(f"TF-IDF shape (sau SMOTE): {X_train_tfidf.shape}")
print(f"Saved: tfidf_vectorizer.pkl")

Vectorizing text với TF-IDF (10K features)...
Áp dụng SMOTE cân bằng dữ liệu TF-IDF...
TF-IDF shape (sau SMOTE): (55419, 10000)
Saved: tfidf_vectorizer.pkl


---
## **Phần 2B: Feature Engineering - Bag-of-Words (BoW)**

Chuyển đổi văn bản thành vector đếm tần suất bằng **CountVectorizer** (Bag-of-Words):

| Tham số | Giá trị | Giải thích |
|---------|---------|------------|
| `max_features` | 50,000 | Giữ lại 50K từ/cụm từ phổ biến nhất |
| `ngram_range` | (1, 2) | Sử dụng cả unigram và bigram |

BoW được dùng riêng cho **Multinomial Naive Bayes** vì mô hình này yêu cầu feature không âm (count-based) thay vì TF-IDF.

In [7]:
print("Vectorizing text với Bag-of-Words (10K features)...")
bow = CountVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_bow_raw = bow.fit_transform(X_train)
X_dev_bow = bow.transform(X_dev)
X_test_bow = bow.transform(X_test)

print("Áp dụng SMOTE cân bằng dữ liệu BoW...")
X_train_bow, y_train_bow = smote.fit_resample(X_train_bow_raw, y_train)

joblib.dump(bow, OUTPUT_DIR / "bow_vectorizer.pkl")

print(f"BoW shape (sau SMOTE): {X_train_bow.shape}")
print(f"Saved: bow_vectorizer.pkl")

Vectorizing text với Bag-of-Words (10K features)...
Áp dụng SMOTE cân bằng dữ liệu BoW...
BoW shape (sau SMOTE): (55419, 10000)
Saved: bow_vectorizer.pkl


---
## **Phần 2C: Feature Engineering - TruncatedSVD**

Áp dụng **TruncatedSVD** để giảm chiều ma trận TF-IDF từ 50,000 features xuống **300 components**.

- Mục đích: giảm RAM và thời gian huấn luyện cho **RandomForest** (vì RF không scale tốt với ma trận thưa chiều cao)
- TruncatedSVD hoạt động tương tự PCA nhưng dùng được trên ma trận sparse
- `n_components=300`: giữ lại 300 chiều có phương sai lớn nhất

In [8]:
print("Giảm chiều TF-IDF bằng TruncatedSVD...")
svd = TruncatedSVD(n_components=300, random_state=RANDOM_STATE)
X_train_svd_raw = svd.fit_transform(X_train_tfidf_raw)  # Dùng raw TF-IDF để giảm chiều
X_dev_svd = svd.transform(X_dev_tfidf)
X_test_svd = svd.transform(X_test_tfidf)

print("Áp dụng SMOTE cân bằng dữ liệu SVD...")
X_train_svd, y_train_svd = smote.fit_resample(X_train_svd_raw, y_train)

joblib.dump(svd, OUTPUT_DIR / "tfidf_svd.pkl")

print(f"SVD shape (sau SMOTE): {X_train_svd.shape}")
print(f"Explained variance ratio (tổng): {svd.explained_variance_ratio_.sum():.4f}")
print(f"Saved: tfidf_svd.pkl")

Giảm chiều TF-IDF bằng TruncatedSVD...
Áp dụng SMOTE cân bằng dữ liệu SVD...
SVD shape (sau SMOTE): (55419, 300)
Explained variance ratio (tổng): 0.3247
Saved: tfidf_svd.pkl


---
## **Khởi tạo Cross-Validation và Dictionary kết quả**

- **StratifiedKFold(n_splits=3)**: chia dữ liệu thành 3 fold, giữ nguyên tỷ lệ phân phối nhãn ở mỗi fold
- Dictionary `results` sẽ lưu toàn bộ metrics của tất cả mô hình, sau đó export ra JSON

In [9]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

results = {
    "metadata": {
        "labels": LABEL_NAMES,
        "n_train": int(len(train_df)),
        "n_dev": int(len(dev_df)),
        "n_test": int(len(test_df)),
    },
    "models": {},
}

---
## **Phần 3A: Huấn luyện Model 1 - Logistic Regression**

**Logistic Regression** là baseline:
- **Hàm mất mát**: Cross-Entropy Loss (log loss)
- **Thuật toán tối ưu**: L-BFGS (Limited-memory Broyden–Fletcher–Goldfarb–Shanno) - hiệu quả cho bài toán multiclass
- **Input**: TF-IDF vectors

**Siêu tham số tìm kiếm (GridSearchCV):**

| Tham số | Giá trị thử | Ý nghĩa |
|---------|-------------|----------|
| `C` | [0.01, 0.1, 1, 10] | Hệ số nghịch đảo regularization - C nhỏ = regularization mạnh |
| `class_weight` | [None, 'balanced'] | None = trọng số đều; 'balanced' = trọng số tỷ lệ nghịch với tần suất lớp |
| `solver` | 'lbfgs' | Thuật toán tối ưu |
| `max_iter` | 1000 | Số iteration tối đa để hội tụ |

In [10]:
print("Training LogisticRegression...")
lr_search = run_grid_search(
    LogisticRegression(max_iter=1000, solver="lbfgs"),
    {"C": [0.01, 0.1, 1, 10], "class_weight": [None, "balanced"]},
    X_train_tfidf,
    y_train_tfidf,
    cv,
)
lr_best = lr_search.best_estimator_
joblib.dump(lr_best, OUTPUT_DIR / "model_lr.pkl")

lr_dev_pred = lr_best.predict(X_dev_tfidf)
lr_test_pred = lr_best.predict(X_test_tfidf)

results["models"]["LogisticRegression"] = {
    "best_params": lr_search.best_params_,
    "best_cv_score": lr_search.best_score_,
    "dev": evaluate_predictions(y_dev, lr_dev_pred),
    "test": evaluate_predictions(y_test, lr_test_pred),
    "confusion_matrix_test": confusion_matrix(
        y_test, lr_test_pred, labels=LABELS
    ).tolist(),
}

print(f"Best params: {lr_search.best_params_}")
print(f"Best CV F1-weighted: {lr_search.best_score_:.4f}")
print(f"Test F1-weighted: {results['models']['LogisticRegression']['test']['f1_weighted']:.4f}")

Training LogisticRegression...
Best params: {'C': 10, 'class_weight': None}
Best CV F1-weighted: 0.8932
Test F1-weighted: 0.7888


---
## **Phần 3B: Huấn luyện Model 2 - Multinomial Naive Bayes**

**Multinomial Naive Bayes** là mô hình xác suất dựa trên định lý Bayes với giả định "naive" (các features độc lập có điều kiện):
- **Hàm mất mát**: Negative Log-Likelihood
- **Thuật toán**: Ước lượng MLE/MAP với Laplace smoothing
- **Input**: BoW vectors (yêu cầu feature không âm, không dùng được TF-IDF với giá trị âm)

**Siêu tham số tìm kiếm (GridSearchCV):**

| Tham số | Giá trị thử | Ý nghĩa |
|---------|-------------|----------|
| `alpha` | [0.1, 0.5, 1.0, 2.0] | Hệ số Laplace smoothing - alpha lớn = smoothing mạnh hơn, giảm overfitting |

In [11]:
print("Training MultinomialNB...")
nb_search = run_grid_search(
    MultinomialNB(),
    {"alpha": [0.1, 0.5, 1.0, 2.0]},
    X_train_bow,
    y_train_bow,
    cv,
)
nb_best = nb_search.best_estimator_
joblib.dump(nb_best, OUTPUT_DIR / "model_nb.pkl")

nb_dev_pred = nb_best.predict(X_dev_bow)
nb_test_pred = nb_best.predict(X_test_bow)

results["models"]["MultinomialNB"] = {
    "best_params": nb_search.best_params_,
    "best_cv_score": nb_search.best_score_,
    "dev": evaluate_predictions(y_dev, nb_dev_pred),
    "test": evaluate_predictions(y_test, nb_test_pred),
    "confusion_matrix_test": confusion_matrix(
        y_test, nb_test_pred, labels=LABELS
    ).tolist(),
}

print(f"Best params: {nb_search.best_params_}")
print(f"Best CV F1-weighted: {nb_search.best_score_:.4f}")
print(f"Test F1-weighted: {results['models']['MultinomialNB']['test']['f1_weighted']:.4f}")

Training MultinomialNB...
Best params: {'alpha': 0.1}
Best CV F1-weighted: 0.6632
Test F1-weighted: 0.8068


---
## **Phần 3C: Huấn luyện Model 3 - Linear SVC (+ CalibratedClassifierCV)**

**Linear SVC** (Support Vector Classifier) tìm siêu phẳng phân tách tối ưu:
- **Hàm mất mát**: Hinge Loss (squared hinge mặc định)
- **Thuật toán tối ưu**: Liblinear (coordinate descent)
- **Input**: TF-IDF vectors

**Lưu ý quan trọng**: LinearSVC không có `predict_proba()`, nên cần wrap bằng **CalibratedClassifierCV** (phương pháp Platt scaling / sigmoid) để có thể:
1. Dùng trong Voting Ensemble (cần xác suất cho soft voting)
2. So sánh confidence scores giữa các mô hình

**Siêu tham số tìm kiếm (GridSearchCV):**

| Tham số | Giá trị thử | Ý nghĩa |
|---------|-------------|----------|
| `C` | [0.1, 1, 5, 10] | Hệ số regularization - C lớn = margin nhỏ, ít regularization |
| `class_weight` | [None, 'balanced'] | Xử lý mất cân bằng lớp |

In [12]:
print("Training LinearSVC (with calibration)...")
svm_search = run_grid_search(
    LinearSVC(),
    {"C": [0.1, 1, 5, 10], "class_weight": [None, "balanced"]},
    X_train_tfidf,
    y_train_tfidf,
    cv,
)
svm_best = svm_search.best_estimator_
svm_calibrated = CalibratedClassifierCV(svm_best, cv=3, method="sigmoid")
svm_calibrated.fit(X_train_tfidf, y_train_tfidf)
joblib.dump(svm_calibrated, OUTPUT_DIR / "model_svm.pkl")

svm_dev_pred = svm_calibrated.predict(X_dev_tfidf)
svm_test_pred = svm_calibrated.predict(X_test_tfidf)

results["models"]["LinearSVC"] = {
    "best_params": svm_search.best_params_,
    "best_cv_score": svm_search.best_score_,
    "dev": evaluate_predictions(y_dev, svm_dev_pred),
    "test": evaluate_predictions(y_test, svm_test_pred),
    "confusion_matrix_test": confusion_matrix(
        y_test, svm_test_pred, labels=LABELS
    ).tolist(),
}

print(f"Best params: {svm_search.best_params_}")
print(f"Best CV F1-weighted: {svm_search.best_score_:.4f}")
print(f"Test F1-weighted: {results['models']['LinearSVC']['test']['f1_weighted']:.4f}")

Training LinearSVC (with calibration)...
Best params: {'C': 10, 'class_weight': 'balanced'}
Best CV F1-weighted: 0.9033
Test F1-weighted: 0.7888


---
## **Phần 3D: Huấn luyện Model 4 - Random Forest**

**Random Forest** là ensemble của nhiều Decision Trees, mỗi cây được huấn luyện trên một tập con ngẫu nhiên của dữ liệu và features:
- **Hàm mất mát**: Gini Impurity (mặc định) - đo độ "không thuần" của node
- **Thuật toán**: Bagging (Bootstrap Aggregating) + random feature selection
- **Input**: TF-IDF + SVD (giảm chiều xuống 300) - vì RF không scale tốt với ma trận sparse 50K chiều

**Siêu tham số tìm kiếm (GridSearchCV):**

| Tham số | Giá trị thử | Ý nghĩa |
|---------|-------------|----------|
| `n_estimators` | [100, 200] | Số lượng cây trong rừng |
| `max_depth` | [10, 30, 50, None] | Độ sâu tối đa của mỗi cây - None = không giới hạn |
| `class_weight` | ['balanced'] | Luôn dùng balanced vì dữ liệu mất cân bằng |

In [13]:
print("Training RandomForest...")
rf_search = run_grid_search(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    {
        "n_estimators": [100, 200],
        "max_depth": [10, 30, 50, None],
        "class_weight": ["balanced"],
    },
    X_train_svd,
    y_train_svd,
    cv,
)
rf_best = rf_search.best_estimator_
joblib.dump(rf_best, OUTPUT_DIR / "model_rf.pkl")

rf_dev_pred = rf_best.predict(X_dev_svd)
rf_test_pred = rf_best.predict(X_test_svd)

results["models"]["RandomForest"] = {
    "best_params": rf_search.best_params_,
    "best_cv_score": rf_search.best_score_,
    "dev": evaluate_predictions(y_dev, rf_dev_pred),
    "test": evaluate_predictions(y_test, rf_test_pred),
    "confusion_matrix_test": confusion_matrix(
        y_test, rf_test_pred, labels=LABELS
    ).tolist(),
}

print(f"Best params: {rf_search.best_params_}")
print(f"Best CV F1-weighted: {rf_search.best_score_:.4f}")
print(f"Test F1-weighted: {results['models']['RandomForest']['test']['f1_weighted']:.4f}")

Training RandomForest...
Best params: {'class_weight': 'balanced', 'max_depth': None, 'n_estimators': 200}
Best CV F1-weighted: 0.9418
Test F1-weighted: 0.8079


---
## **Phần 3E: Huấn luyện Model 5 - SGD Classifier (Bước 1: GridSearch tìm siêu tham số)**

**SGD Classifier** (Stochastic Gradient Descent) là mô hình tuyến tính huấn luyện bằng gradient descent ngẫu nhiên:
- **Hàm mất mát**: Log Loss (tương đương Cross-Entropy, giống Logistic Regression)
- **Thuật toán tối ưu**: SGD - cập nhật trọng số từng mini-batch, hiệu quả trên dữ liệu lớn
- **Input**: TF-IDF vectors

SGD được huấn luyện qua **2 bước**:
1. **Bước 1 (cell này)**: GridSearchCV để tìm siêu tham số tốt nhất (`alpha`, `class_weight`)
2. **Bước 2 (cell tiếp theo)**: Train lại với `partial_fit` theo từng epoch để log loss/accuracy

**Siêu tham số tìm kiếm:**

| Tham số | Giá trị thử | Ý nghĩa |
|---------|-------------|----------|
| `alpha` | [1e-4, 1e-3] | Hệ số regularization L2 |
| `class_weight` | [None, 'balanced'] | Xử lý mất cân bằng lớp |

In [14]:
print("Training SGDClassifier (grid search)...")
sgd_search = run_grid_search(
    SGDClassifier(loss="log_loss", max_iter=1000, tol=1e-3, random_state=RANDOM_STATE),
    {"alpha": [1e-4, 1e-3], "class_weight": [None, "balanced"]},
    X_train_tfidf,
    y_train_tfidf,
    cv,
)
sgd_best_params = sgd_search.best_params_

print(f"Best params: {sgd_best_params}")
print(f"Best CV F1-weighted: {sgd_search.best_score_:.4f}")

Training SGDClassifier (grid search)...
Best params: {'alpha': 0.0001, 'class_weight': None}
Best CV F1-weighted: 0.7452


---
## **Phần 3F: Huấn luyện Model 5 - SGD Classifier (Bước 2: Epoch Logging với partial_fit)**

Sau khi tìm được siêu tham số tốt nhất từ GridSearch, huấn luyện lại SGD bằng **`partial_fit`** theo từng epoch (20 epochs) để:
- Ghi nhận **loss** (log loss) và **accuracy** trên cả train/dev sau mỗi epoch
- Vẽ **Epoch Curves** (yêu cầu bắt buộc của đề bài) - thể hiện quá trình hội tụ của mô hình

**Chi tiết kỹ thuật:**
- Lỗi `class_weight='balanced'` không hỗ trợ `partial_fit` được xử lý bằng cách tính thủ công `compute_class_weight`.
- Mỗi epoch: shuffle dữ liệu train (`rng.permutation`) rồi gọi `partial_fit`
- Ghi nhận 5 metrics sau mỗi epoch: `train_loss`, `val_loss`, `train_acc`, `val_acc`, `train_f1`, `val_f1`
- Kết quả lưu vào `sgd_epoch_curve.json` để file 07 vẽ biểu đồ

In [15]:
print("Training SGDClassifier (epoch logging)...")
cw_setting = sgd_best_params["class_weight"]
if cw_setting == "balanced":
    weights = compute_class_weight("balanced", classes=LABELS, y=y_train_tfidf)
    cw_dict = {label: weight for label, weight in zip(LABELS, weights)}
else:
    cw_dict = cw_setting

sgd_epochs = 20
sgd_epoch = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    alpha=sgd_best_params["alpha"],
    class_weight=cw_dict,
    learning_rate="optimal",
    random_state=RANDOM_STATE,
    max_iter=1,
    tol=None,
)

rng = np.random.default_rng(RANDOM_STATE)
sgd_history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": [],
    "train_f1": [],
    "val_f1": [],
}

for epoch in range(1, sgd_epochs + 1):
    perm = rng.permutation(len(y_train_tfidf))
    sgd_epoch.partial_fit(X_train_tfidf[perm], y_train_tfidf[perm], classes=LABELS)

    train_proba = sgd_epoch.predict_proba(X_train_tfidf)
    dev_proba = sgd_epoch.predict_proba(X_dev_tfidf)
    train_pred = train_proba.argmax(axis=1)
    dev_pred = dev_proba.argmax(axis=1)

    sgd_history["epoch"].append(epoch)
    sgd_history["train_loss"].append(
        log_loss(y_train_tfidf, train_proba, labels=LABELS)
    )
    sgd_history["val_loss"].append(log_loss(y_dev, dev_proba, labels=LABELS))
    sgd_history["train_acc"].append(accuracy_score(y_train_tfidf, train_pred))
    sgd_history["val_acc"].append(accuracy_score(y_dev, dev_pred))
    sgd_history["train_f1"].append(
        f1_score(y_train_tfidf, train_pred, average="weighted", zero_division=0)
    )
    sgd_history["val_f1"].append(
        f1_score(y_dev, dev_pred, average="weighted", zero_division=0)
    )
    print(f"  Epoch {epoch:2d} | Train Loss: {sgd_history['train_loss'][-1]:.4f} | Val Loss: {sgd_history['val_loss'][-1]:.4f} | Val Acc: {sgd_history['val_acc'][-1]:.4f}")

save_json(OUTPUT_DIR / "sgd_epoch_curve.json", sgd_history)
joblib.dump(sgd_epoch, OUTPUT_DIR / "model_sgd.pkl")

sgd_dev_pred = sgd_epoch.predict(X_dev_tfidf)
sgd_test_pred = sgd_epoch.predict(X_test_tfidf)

results["models"]["SGDClassifier"] = {
    "best_params": sgd_best_params,
    "best_cv_score": sgd_search.best_score_,
    "dev": evaluate_predictions(y_dev, sgd_dev_pred),
    "test": evaluate_predictions(y_test, sgd_test_pred),
    "confusion_matrix_test": confusion_matrix(
        y_test, sgd_test_pred, labels=LABELS
    ).tolist(),
}

print(f"\nTest F1-weighted: {results['models']['SGDClassifier']['test']['f1_weighted']:.4f}")
print(f"Saved: model_sgd.pkl, sgd_epoch_curve.json")

Training SGDClassifier (epoch logging)...
  Epoch  1 | Train Loss: 0.7174 | Val Loss: 0.7149 | Val Acc: 0.7387
  Epoch  2 | Train Loss: 0.7192 | Val Loss: 0.7233 | Val Acc: 0.7364
  Epoch  3 | Train Loss: 0.7177 | Val Loss: 0.7217 | Val Acc: 0.7368
  Epoch  4 | Train Loss: 0.7197 | Val Loss: 0.7203 | Val Acc: 0.7398
  Epoch  5 | Train Loss: 0.7189 | Val Loss: 0.7182 | Val Acc: 0.7406
  Epoch  6 | Train Loss: 0.7192 | Val Loss: 0.7215 | Val Acc: 0.7372
  Epoch  7 | Train Loss: 0.7189 | Val Loss: 0.7237 | Val Acc: 0.7357
  Epoch  8 | Train Loss: 0.7189 | Val Loss: 0.7223 | Val Acc: 0.7364
  Epoch  9 | Train Loss: 0.7190 | Val Loss: 0.7253 | Val Acc: 0.7349
  Epoch 10 | Train Loss: 0.7191 | Val Loss: 0.7246 | Val Acc: 0.7357
  Epoch 11 | Train Loss: 0.7190 | Val Loss: 0.7234 | Val Acc: 0.7357
  Epoch 12 | Train Loss: 0.7193 | Val Loss: 0.7236 | Val Acc: 0.7357
  Epoch 13 | Train Loss: 0.7191 | Val Loss: 0.7230 | Val Acc: 0.7357
  Epoch 14 | Train Loss: 0.7194 | Val Loss: 0.7237 | Val Acc:

---
## **Phần 3G: Huấn luyện Model 6 - Voting Ensemble (Soft Voting)**

**Voting Ensemble** kết hợp dự đoán của nhiều mô hình bằng cách **trung bình xác suất** (soft voting):

$$\hat{y} = \arg\max_c \frac{1}{K} \sum_{k=1}^{K} P_k(y = c | x)$$

Với $K = 3$ mô hình thành viên:
1. **Logistic Regression** (best từ GridSearch)
2. **Linear SVC** (calibrated - đã có `predict_proba` nhờ CalibratedClassifierCV)
3. **SGD Classifier** (đã train bằng `partial_fit`)

**Không cần GridSearch** cho Voting vì nó sử dụng các mô hình đã được tối ưu riêng.

**Tại sao không dùng `sklearn.ensemble.VotingClassifier`?** Vì SGD đã train bằng `partial_fit`, ta cần dùng trực tiếp mô hình đã train thay vì re-train.

In [16]:
print("Training Voting ensemble (soft voting)...")
voting_models = [lr_best, svm_calibrated, sgd_epoch]

voting_dev_proba = soft_voting_predict_proba(voting_models, X_dev_tfidf)
voting_test_proba = soft_voting_predict_proba(voting_models, X_test_tfidf)
voting_dev_pred = voting_dev_proba.argmax(axis=1)
voting_test_pred = voting_test_proba.argmax(axis=1)

results["models"]["VotingEnsemble"] = {
    "best_params": {
        "members": ["LogisticRegression", "LinearSVC(calibrated)", "SGDClassifier"],
        "strategy": "mean_proba",
    },
    "best_cv_score": None,
    "dev": evaluate_predictions(y_dev, voting_dev_pred),
    "test": evaluate_predictions(y_test, voting_test_pred),
    "confusion_matrix_test": confusion_matrix(
        y_test, voting_test_pred, labels=LABELS
    ).tolist(),
}

joblib.dump(
    {"lr": lr_best, "svm": svm_calibrated, "sgd": sgd_epoch},
    OUTPUT_DIR / "model_voting.pkl",
)

print(f"Test F1-weighted: {results['models']['VotingEnsemble']['test']['f1_weighted']:.4f}")
print(f"Saved: model_voting.pkl")

Training Voting ensemble (soft voting)...
Test F1-weighted: 0.7921
Saved: model_voting.pkl


---
## **Phần 4: Learning Curves (theo Training Set Size)**

Vẽ **Learning Curves** cho 3 mô hình đại diện (LR, SVM, RF) để phân tích **bias/variance**:

- Dùng `sklearn.model_selection.learning_curve()` với:
  - `train_sizes = np.linspace(0.1, 1.0, 10)` - 10 mức từ 10% đến 100% dữ liệu train
  - `scoring = 'f1_weighted'` - metric đánh giá
  - `cv = 3` - 3-fold cross-validation

**Cách đọc Learning Curve:**
- **Train score cao, Val score thấp** → Overfitting (high variance)
- **Cả hai score thấp** → Underfitting (high bias)
- **Hai đường hội tụ ở mức cao** → Mô hình tốt

Kết quả lưu vào `learning_curve_data.json` để file 07 vẽ biểu đồ.

In [ ]:
print("Computing learning curves...")
learning_curve_data = {}
train_sizes = np.linspace(0.1, 1.0, 10)

# Learning curve for Logistic Regression
print("  - Logistic Regression...")
lr_curve = learning_curve(
    clone(lr_best),
    X_train_tfidf,
    y_train_tfidf,
    cv=cv,
    scoring="f1_weighted",
    train_sizes=train_sizes,
    n_jobs=-1,
)
learning_curve_data["LogisticRegression"] = {
    "train_sizes": lr_curve[0].tolist(),
    "train_mean": lr_curve[1].mean(axis=1).tolist(),
    "train_std": lr_curve[1].std(axis=1).tolist(),
    "val_mean": lr_curve[2].mean(axis=1).tolist(),
    "val_std": lr_curve[2].std(axis=1).tolist(),
}

# Learning curve for Linear SVC
print("  - Linear SVC...")
svm_curve_estimator = LinearSVC(**svm_search.best_params_)
svm_curve = learning_curve(
    svm_curve_estimator,
    X_train_tfidf,
    y_train_tfidf,
    cv=cv,
    scoring="f1_weighted",
    train_sizes=train_sizes,
    n_jobs=-1,
)
learning_curve_data["LinearSVC"] = {
    "train_sizes": svm_curve[0].tolist(),
    "train_mean": svm_curve[1].mean(axis=1).tolist(),
    "train_std": svm_curve[1].std(axis=1).tolist(),
    "val_mean": svm_curve[2].mean(axis=1).tolist(),
    "val_std": svm_curve[2].std(axis=1).tolist(),
}

# Learning curve for Random Forest
print("  - Random Forest...")
rf_curve = learning_curve(
    clone(rf_best),
    X_train_svd,
    y_train_tfidf,
    cv=cv,
    scoring="f1_weighted",
    train_sizes=train_sizes,
    n_jobs=-1,
)
learning_curve_data["RandomForest"] = {
    "train_sizes": rf_curve[0].tolist(),
    "train_mean": rf_curve[1].mean(axis=1).tolist(),
    "train_std": rf_curve[1].std(axis=1).tolist(),
    "val_mean": rf_curve[2].mean(axis=1).tolist(),
    "val_std": rf_curve[2].std(axis=1).tolist(),
}

save_json(OUTPUT_DIR / "learning_curve_data.json", learning_curve_data)
print("Saved: learning_curve_data.json")

Computing learning curves...
  - Logistic Regression...
  - Linear SVC...
  - Random Forest...
Saved: learning_curve_data.json


---
## **Phần 5: Error Analysis (Phân tích lỗi)**

Phân tích chi tiết các mẫu mà mô hình tốt nhất (**Voting Ensemble**) dự đoán sai trên tập test:

1. **Lưu predictions**: Tạo CSV chứa `free_text_clean`, `y_true`, `y_pred`, tên nhãn - để có thể review lại từng mẫu
2. **Phân nhóm lỗi**: Nhóm theo cặp `(y_true → y_pred)` để xem loại nhầm nào phổ biến nhất (ví dụ: OFFENSIVE bị nhầm thành CLEAN)
3. **Ví dụ cụ thể**: Lấy 5 ví dụ text cho mỗi loại nhầm phổ biến nhất (top 5 loại nhầm)
4. **Lưu JSON**: Export toàn bộ phân tích ra `error_analysis.json`

Mục đích: Giúp hiểu **tại sao** mô hình sai, từ đó đề xuất cải thiện trong báo cáo.

In [18]:
print("Saving test predictions and error analysis...")
pred_df = pd.DataFrame(
    {
        "free_text_clean": X_test,
        "y_true": y_test,
        "y_pred": voting_test_pred,
    }
)
pred_df["y_true_label"] = pred_df["y_true"].map(LABEL_NAMES)
pred_df["y_pred_label"] = pred_df["y_pred"].map(LABEL_NAMES)
pred_df.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False, encoding="utf-8")

errors = pred_df[pred_df["y_true"] != pred_df["y_pred"]]
conf_counts = (
    errors.groupby(["y_true", "y_pred"]).size().sort_values(ascending=False)
)

error_examples = []
for (y_true_val, y_pred_val), count in conf_counts.head(5).items():
    subset = errors[
        (errors["y_true"] == y_true_val) & (errors["y_pred"] == y_pred_val)
    ].head(5)
    samples = [
        {
            "text": str(text)[:200],
            "y_true": int(y_true_val),
            "y_pred": int(y_pred_val),
            "y_true_label": LABEL_NAMES[int(y_true_val)],
            "y_pred_label": LABEL_NAMES[int(y_pred_val)],
        }
        for text in subset["free_text_clean"].tolist()
    ]
    error_examples.append(
        {
            "y_true": int(y_true_val),
            "y_pred": int(y_pred_val),
            "count": int(count),
            "examples": samples,
        }
    )

error_analysis = {
    "total_errors": int(len(errors)),
    "top_confusions": error_examples,
    "confusion_counts": {
        f"{int(k[0])}->{int(k[1])}": int(v) for k, v in conf_counts.items()
    },
}

save_json(OUTPUT_DIR / "error_analysis.json", error_analysis)

print(f"Tổng số mẫu dự đoán sai: {len(errors)} / {len(pred_df)} ({len(errors)/len(pred_df)*100:.1f}%)")
print(f"\nTop 5 loại nhầm phổ biến nhất:")
for item in error_examples:
    print(f"  {LABEL_NAMES[item['y_true']]} -> {LABEL_NAMES[item['y_pred']]}: {item['count']} mẫu")
print(f"\nSaved: test_predictions.csv, error_analysis.json")

Saving test predictions and error analysis...
Tổng số mẫu dự đoán sai: 1468 / 6527 (22.5%)

Top 5 loại nhầm phổ biến nhất:
  CLEAN -> HATE: 474 mẫu
  CLEAN -> OFFENSIVE: 402 mẫu
  HATE -> CLEAN: 263 mẫu
  OFFENSIVE -> CLEAN: 158 mẫu
  OFFENSIVE -> HATE: 100 mẫu

Saved: test_predictions.csv, error_analysis.json


---
## **Phần 6: Tổng hợp & Lưu kết quả**

Tổng hợp kết quả của tất cả 6 mô hình thành bảng so sánh:

| Cột | Ý nghĩa |
|-----|----------|
| `model_name` | Tên mô hình |
| `dev_acc` | Accuracy trên tập dev |
| `dev_f1w` | F1-weighted trên tập dev |
| `test_acc` | Accuracy trên tập test |
| `test_f1w` | F1-weighted trên tập test |
| `test_f1macro` | F1-macro trên tập test (quan trọng khi dữ liệu mất cân bằng) |

**Output:**
- `all_results.json`: Toàn bộ metrics chi tiết (bao gồm classification report, confusion matrix)
- `all_results.csv`: Bảng tóm tắt dạng CSV để dùng trong báo cáo

In [19]:
summary_rows = []
for model_name, info in results["models"].items():
    summary_rows.append(
        {
            "model_name": model_name,
            "dev_acc": info["dev"]["accuracy"],
            "dev_f1w": info["dev"]["f1_weighted"],
            "test_acc": info["test"]["accuracy"],
            "test_f1w": info["test"]["f1_weighted"],
            "test_f1macro": info["test"]["f1_macro"],
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / "all_results.csv", index=False, encoding="utf-8")

save_json(OUTPUT_DIR / "all_results.json", results)

print("=== MODEL SUMMARY (TEST F1-WEIGHTED) ===")
print(summary_df[["model_name", "test_f1w"]].sort_values(by="test_f1w", ascending=False).to_string(index=False))
print(f"\nArtifacts saved to: {OUTPUT_DIR}")

=== MODEL SUMMARY (TEST F1-WEIGHTED) ===
        model_name  test_f1w
      RandomForest  0.807880
     MultinomialNB  0.806785
    VotingEnsemble  0.792139
LogisticRegression  0.788820
         LinearSVC  0.788764
     SGDClassifier  0.774560

Artifacts saved to: C:\Users\ADMIN\Desktop\I2ML\Intro2MLFinal\output\model
